In [227]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier # vecinos más cercanos para clasificación
from sklearn.metrics import accuracy_score # métrica de evaluación
from sklearn.metrics import classification_report


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS
from sklearn.feature_selection import SequentialFeatureSelector, RFECV, SelectFromModel
from sklearn.model_selection import GridSearchCV
import time

from sklearn import set_config
set_config(transform_output="pandas")

In [228]:

def get_bmi(X, eps):
    height_m = X['height(cm)'] / 100.0
    return X['weight(kg)'] / (height_m ** 2 + eps)

#Multicolumn FE. Si fuese sobre una unica columna usar lambdas o cosas mas simples,
#Desarrollado el 25/09/2026, sujeto a pruebas
class MCFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, new_feature_name="", drop_features=False,  operation=None):
        self.operation          = operation
        self.new_feature_name   = new_feature_name
        self.drop_features      = drop_features
        self.eps                = 1e-6 #Para evitar que explote en divisiones

    def fit(self, X, y=None):
        self.current_features = X.columns
        return self

    def get_feature_name_out(self, input_features=None):
        if self.drop_features:
            return np.array([self.new_feature_name])
        else:
            if input_features is not None:
                # Retorna las columnas originales más la nueva
                return np.array(list(input_features) + [self.new_feature_name])
            return np.array([self.new_feature_name])
        

    def transform(self, X):
        if self.operation == None:
            raise Exception("No especifico la funcion") # ponerlo en ingles!

        Xc = X.copy()
        Xc[self.new_feature_name] = self.operation(X, self.eps)
        if self.drop_features:
            Xc.drop(columns=self.current_features, inplace=True)
        return Xc


'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)',
       'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic',
       'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride',
       'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST',
       'ALT', 'Gtp', 'dental caries', 'smoking'

Como en progra 1, pruebo individualmente a ver si funciona!

In [229]:
df_smokers = pd.read_csv('https://raw.githubusercontent.com/pokengineer/DataScience/main/datasets/smokers.csv')
fe = MCFeatureEngineer(new_feature_name="BMI", drop_features=False, operation=get_bmi)
fe.fit(df_smokers[["height(cm)", "weight(kg)"]])
fe.transform(df_smokers[["height(cm)", "weight(kg)"]])

,height(cm),weight(kg),BMI
0,170,85,29.411755
1,175,110,35.918356
2,155,65,27.055140
3,165,80,29.384746
4,165,60,22.038559
...,...,...,...
38979,165,60,22.038559
38980,155,55,22.892810
38981,170,105,36.332167
38982,160,55,21.484367


Listo, si funciona lo uso para todo lo que necesite.

In [230]:
def get_WHtR(X, eps):
    return X['Cholesterol'] / (X['HDL'] + eps)
def get_TG_HDL_Ratio(X, eps):
    return X['triglyceride'] / (X['HDL'] + eps) 
def get_Non_HDL_Cholesterol(X, eps):
    return X['Cholesterol'] - X['HDL']
#Presion
def get_Pulse_Pressure(X, eps):
    return X['systolic'] - X['relaxation']
def get_MAP(X, eps):
    # Este es ligeramente distino, necesita de Pulse_Pressure que debe ser calculado previamente!!
    return  X['relaxation'] + (X['Pulse_Pressure'] / 3.0)

#pl_pressure = Pipeline(steps=[("Pulse_Pressure", MCFeatureEngineer(new_feature_name="Pulse_Pressure", drop_features=False,  operation=get_Pulse_Pressure)),
#                              ("MAP", MCFeatureEngineer(new_feature_name="MAP", drop_features=False,  operation=get_MAP))])

#Si le pongo False en todos los drop_features=False, se produce repeticion de columnas, no quiero verbose_feature_names_out=True que puede
#producir colinealidad
ct1 = ColumnTransformer(
        transformers=[
            ("BMI", MCFeatureEngineer(new_feature_name="BMI", drop_features=True,  operation=get_bmi), ['height(cm)', 'weight(kg)']),
            ("WHtR", MCFeatureEngineer(new_feature_name="WHtR", drop_features=True,  operation=get_WHtR), ['Cholesterol', 'HDL']),
            ("TG_HDL_Ratio", MCFeatureEngineer(new_feature_name="TG_HDL_Ratio", drop_features=True,  operation=get_TG_HDL_Ratio), ['triglyceride', 'HDL']),
            ("Non_HDL_Cholesterol", MCFeatureEngineer(new_feature_name="Non_HDL_Cholesterol", drop_features=True,  operation=get_Non_HDL_Cholesterol), ['Cholesterol', 'HDL']),
            ("Pulse_Pressure", MCFeatureEngineer(new_feature_name="Pulse_Pressure", drop_features=False,  operation=get_Pulse_Pressure), ['systolic','relaxation']) 
        ],
        verbose_feature_names_out=False,
        remainder="passthrough"
    )

ct2 = ColumnTransformer(
    transformers=[
        ("MAP", MCFeatureEngineer(new_feature_name="MAP", drop_features=True,  operation=get_MAP), ["Pulse_Pressure","relaxation"])
    ],
    verbose_feature_names_out=False,
    remainder="passthrough"
)


In [231]:
ct2.fit_transform(ct1.fit_transform(df_smokers))

,MAP,BMI,WHtR,TG_HDL_Ratio,Non_HDL_Cholesterol,systolic,age,waist(cm),eyesight(left),eyesight(right),...,fasting blood sugar,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,smoking
0,91.333333,29.411755,3.414286,2.185714,169,118,35,97.0,0.9,0.9,...,97,142,19.8,1,1.0,61,115,125,1,1
1,92.333333,35.918356,2.971831,1.802817,140,119,20,110.0,0.7,0.9,...,88,114,15.9,1,1.1,19,25,30,1,0
2,90.000000,27.055140,3.385965,2.105263,136,110,45,86.0,0.9,0.9,...,80,112,13.7,3,0.6,1090,1400,276,0,0
3,111.333333,29.384746,4.565217,7.956522,164,158,45,94.0,0.8,0.7,...,249,91,16.9,1,0.9,32,36,36,0,0
4,79.000000,22.038559,3.808511,4.255319,132,109,20,81.0,1.5,0.1,...,100,92,14.9,1,1.2,26,28,15,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38979,75.666667,22.038559,2.360656,0.868852,83,107,40,80.0,0.4,0.6,...,93,72,12.3,1,0.5,18,18,21,1,0
38980,90.000000,22.892810,2.986842,1.315789,151,126,45,75.0,1.5,1.2,...,91,131,12.5,2,0.6,23,11,12,0,0
38981,103.666667,36.332167,4.687500,4.083333,177,141,40,124.0,0.6,0.5,...,115,138,17.1,1,0.8,24,23,35,1,1
38982,77.666667,21.484367,2.607595,0.607595,127,95,40,75.0,1.5,1.5,...,102,116,12.0,1,0.6,24,20,17,0,1


In [232]:
X = df_smokers[df_smokers.columns.drop("smoking")]
y = df_smokers["smoking"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify = y, random_state=42)

pl = Pipeline(steps=[
    ("CT1", ct1),
    ("CT2", ct2),
    ("scaler", MinMaxScaler())    
])



In [233]:
pl.fit(X_train)
X_train_tr = pl.transform(X_train)
X_test_tr  = pl.transform(X_test)

In [234]:
sfs = SequentialFeatureSelector(KNeighborsClassifier(n_neighbors=11), n_features_to_select=3, scoring="accuracy", cv=5, direction="backward", n_jobs=-1)
sfs.fit(X_train_tr, y_train)
sfs.get_feature_names_out()

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

array(['age', 'hemoglobin', 'Gtp'], dtype=object)

,MAP,BMI,WHtR,TG_HDL_Ratio,Non_HDL_Cholesterol,systolic,age,waist(cm),eyesight(left),eyesight(right),...,hearing(right),fasting blood sugar,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries
32278,0.296496,0.533286,0.216859,0.070595,0.435484,0.240741,0.615385,0.612777,0.051020,0.020408,...,0.0,0.122016,0.038695,0.592593,0.0,0.052174,0.010157,0.003779,0.029087,0.0
7972,0.371968,0.463978,0.166467,0.045559,0.442652,0.388889,0.307692,0.456323,0.091837,0.091837,...,0.0,0.119363,0.040354,0.506173,0.0,0.052174,0.008310,0.003435,0.017051,0.0
22602,0.247978,0.420639,0.477785,0.205145,0.589606,0.216049,0.307692,0.443286,0.081633,0.112245,...,0.0,0.095491,0.076285,0.660494,0.0,0.060870,0.016620,0.012710,0.043129,0.0
21667,0.347709,0.230393,0.200374,0.033407,0.467742,0.345679,0.230769,0.372881,0.112245,0.091837,...,0.0,0.122016,0.050857,0.524691,0.0,0.086957,0.010157,0.004466,0.024072,0.0
32395,0.498652,0.353181,0.195879,0.026623,0.473118,0.481481,0.076923,0.391134,0.040816,0.061224,...,0.0,0.108753,0.042565,0.641975,0.0,0.086957,0.023084,0.007214,0.026078,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
659,0.280323,0.475969,0.211792,0.173652,0.519713,0.240741,0.461538,0.456323,0.112245,0.112245,...,0.0,0.249337,0.042565,0.641975,0.0,0.086957,0.007387,0.009619,0.055165,1.0
4752,0.177898,0.282236,0.174683,0.039040,0.453405,0.179012,0.307692,0.323338,0.081633,0.081633,...,0.0,0.092838,0.044776,0.462963,0.0,0.034783,0.006464,0.002061,0.009027,1.0
30885,0.323450,0.315094,0.212275,0.043817,0.532258,0.388889,0.384615,0.456323,0.142857,0.142857,...,0.0,0.103448,0.067441,0.623457,0.0,0.104348,0.010157,0.005496,0.042126,0.0
31028,0.318059,0.256053,0.357033,0.097040,0.614695,0.339506,0.692308,0.456323,0.061224,0.081633,...,0.0,0.480106,0.088447,0.635802,0.0,0.069565,0.013850,0.004122,0.010030,0.0


In [ ]:
#Agrego el estimador
param_grid = {
    "clf__k":[5,9,15,31]
}

pl.steps.append(("clf", KNeighborsClassifier(n_neighbors=k)))
gs = GridSearchCV(estimator=pl, param_grid=param_grid, scoring="accuracy" cv=5, n_jobs=-1)
gs.fit(X_train_tr, y_train)
gs.best_estimator_

In [ ]:
min_features = 3
max_features = 3
knn = KNeighborsClassifier(n_neighbors=11)

efs = EFS(
    knn, 
    min_features=min_features, 
    max_features=max_features, # Evalúa hasta el total de columnas
    scoring='accuracy',
    cv=5,
    print_progress=True
)

efs = efs.fit(X_train_tr, y_train)

print('Mejor combinación de índices:', efs.best_idx_)
print('Mejor puntaje:', efs.best_score_)

X_train_efs = efs.transform(X_train)
X_test_efs = efs.transform(X_test)

Features: 1540/1540

Mejor combinación de índices: (0, 1, 20)
Mejor puntaje: 0.7156258399977491


In [216]:
X_train

,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,triglyceride,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries
32278,60,160,75,98.0,0.6,0.3,1,1,110,77,...,72,50,71,14.5,1,0.7,18,14,31,0
7972,40,160,70,86.0,1.0,1.0,1,1,134,79,...,72,74,74,13.1,1,0.7,16,13,19,0
22602,40,175,80,85.0,0.9,1.2,1,1,106,70,...,158,40,139,15.6,1,0.8,25,40,45,0
21667,35,170,60,79.6,1.2,1.0,1,1,127,78,...,50,67,93,13.4,1,1.1,18,16,26,0
32395,25,170,70,81.0,0.5,0.7,1,1,149,95,...,44,71,78,15.3,1,1.1,32,24,28,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
659,50,170,80,86.0,1.2,1.2,1,1,110,74,...,269,80,78,15.3,1,1.1,15,31,57,1
4752,40,150,50,75.8,0.9,0.9,1,1,100,60,...,63,74,82,12.4,1,0.5,14,9,11,1
30885,45,180,75,86.0,1.5,1.5,1,1,134,70,...,79,84,123,15.0,1,1.3,18,19,44,0
31028,65,160,55,86.0,0.7,0.9,1,1,126,73,...,116,60,161,15.2,1,0.9,22,15,12,0


In [ ]:
X_train.iloc[:, list(efs.best_idx_)]

(0, 1, 20)

,age,height(cm),Gtp
32278,60,160,31
7972,40,160,19
22602,40,175,45
21667,35,170,26
32395,25,170,28
...,...,...,...
659,50,170,57
4752,40,150,11
30885,45,180,44
31028,65,160,12


NameError: name 'y_train_tr' is not defined

array(['age', 'height(cm)', 'Gtp'], dtype=object)

In [ ]:
"""
X['Remnant_Cholesterol'] = X['Cholesterol'] - X['HDL'] - X['LDL']
# 3. Presión Arterial


# 4. Enzimas Hepáticas
X['AST_ALT_Ratio'] = X['AST'] / (X['ALT'] + self.eps)

# 5. Visión y Audición
X['eyesight_mean'] = (X['eyesight(left)'] + X['eyesight(right)']) / 2.0
X['eyesight_diff'] = (X['eyesight(left)'] - X['eyesight(right)']).abs()
X['hearing_any_loss'] = ((X['hearing(left)'] == 2) | (X['hearing(right)'] == 2)).astype(int)
"""

# Análisis Exploratorio

In [ ]:
df_smokers = pd.read_csv('https://raw.githubusercontent.com/pokengineer/DataScience/main/datasets/smokers.csv')
df_smokers.head(5)

1.Ya sabemos de practicas anteriores que no tiene nulos y ademas los tipos son int y float. Desde ese punto de vista los datos son sanos y no requieren intervención.  
2.Tambien sabemos que hay variables con alta tasa de correlación.

Arranquemos con ingenieria de caracteristicas.  
IMC / BMI : Peso en kilos / (Altura en cm / 100)**2  
Cintura-Estatura: waist(cm)/height(cm)
Colesterol total: Colesterol / HDL  
Trigliceridos HDL : Trigriceridos / HDL  
Colesterol no hdl : Colesterol - HDL  
Remanente lipidos: Colesterol - HDL - LDL  
Sistolica: relaxation + ((Systolic-relaxation)/3)  
Presion pulso: Sistolica - relaxation   
Cociente de De Ritis: AST/ALT  
Core Hepático Combinado: PCA para Gtp, AST y ALT.
Vision promedio y diferencia, ojos iz + der / 2 y abs(iz-der)
 



In [ ]:
df_smokers.columns

Vamos con las transformaciones pero ¿Function transformers o Custom Transformers? Ambas son válidas. Criterio personal, transformaciones simples FunctionTransformer+lambda. Transformaciones 1 a 1, es una cuenta sobre el registro que no involucra al resto del dataset 


In [ ]:
ct_bmi = ("BMI", FunctionTransformer() )

# Correlación de Variables

In [ ]:
plt.figure(figsize=(14,7))
sns.heatmap(df_smokers.corr(), annot=True, vmax=.7, cmap ='Blues', fmt=".2f")

In [ ]:
df_smokers_corr = df_smokers.corr()[["smoking"]]*100 # lo pasamos a porcentajes
df_smokers_corr = df_smokers_corr.drop("smoking", axis=0) # eliminamos la variable target
df_smokers_corr = df_smokers_corr.sort_values(["smoking"], ascending=False) # ordenamos en forma descendente
df_smokers_corr = abs(df_smokers_corr) # nos interesa el valor absouluto
df_smokers_corr

# Seleccionamos y Escalamos las variables que vamos a utilizar

In [ ]:
# definimos df con las 10 columnas que elegimos para el modelo
# sería lo mismo que hacer un drop de las que no queremos
df = df_smokers[['hemoglobin', 'height(cm)', 'weight(kg)', 'triglyceride', 'Gtp',
       'waist(cm)', 'serum creatinine', 'dental caries', 'relaxation',
       'fasting blood sugar','smoking']]

In [ ]:
# Hacemos el Split 70-30 para train-test
y_smokers = df["smoking"]
X_train, X_test, y_train, y_test = train_test_split(df.drop(["smoking"],axis = 1), y_smokers, test_size=0.3, stratify = y_smokers, random_state=42)

In [ ]:
# usamos StandardScaler para escalar las variables
scaler_X = StandardScaler(with_mean=True, with_std=True)
scaler_X.fit(X_train) # entrenamos los valores quitandole la variable clase

In [ ]:
X_train = scaler_X.transform(X_train)
X_test = scaler_X.transform(X_test)

In [ ]:
test_scores = []

# Creamos y entrenamos el algoritmo con 20 valores de K
for k in range(3,40,2):
  knn = KNeighborsClassifier(k)
  knn.fit(X_train,y_train) # Creamos y entrenamos el clasificador knn

  # Para cada valor de K, evaluamos la capacidad de clasificación con datos de prueba
  y_pred = knn.predict(X_test)
  test_scores.append(accuracy_score(y_test, y_pred)) # Agregamos los K resultados de evaluación

In [ ]:
df_scores = pd.DataFrame([{"k":valor_k, "score":test_scores_k} for valor_k, test_scores_k in zip(range(3,40,2),test_scores)])
plt.plot(df_scores["k"], df_scores["score"])

In [ ]:
df_scores

In [ ]:
# Entrenamos el algoritmo con el mejor K
knn = KNeighborsClassifier( 11 )
knn.fit(X_train,y_train) # Entrenamos el clasificador

In [ ]:
y_pred_knn = knn.predict(X_test)

#Exactitud del modelo
print('Exactitud (accuracy) del modelo: {:.2f} %'.format(accuracy_score(y_test, y_pred_knn)*100))
print("-"*100)

# Reporte del clasificador
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred_knn))

# Ejercicio
Dado el análisis exploratorio concluimos que no hace falta imputar variables y que la variable target esta balanceada.
- Usando el mapa de calor de correlaciones, ¿que pares de columnas comparten más de un 0.7 de correlación?
- Entrenar un modelo con K menor a 11, comparar los resultados
- Entrenar un modelo con K mayor a 11, comparar los resultados
- Entrenar un modelo con 3 variables, seleccionar un K apropiado y comparar los resultados
- Entrenar un modelo con todas las variables, seleccionar un K apropiado y comparar los resultados
- "Feature Engineering", Crear una columna nueva no redundante, entrenar un modelo y comparar los resultados

Uso de MLExtend en remplazo de SelectFromModel/SequentialFeatureSelector o metodos de filtados (univariables) como SelectKBest.

Me puedo dar este lujo porque el dataset es pequeño. Aprovecho tambien la oportunidad para aplicar maxmin en lugar de StandardScaler

In [ ]:

from sklearn.preprocessing import MinMaxScaler
X_train, X_test, y_train, y_test = train_test_split(df.drop(["smoking"],axis = 1), df["smoking"], test_size=0.3, stratify = y_smokers, random_state=42)
scaler_X = MinMaxScaler(feature_range=(0, 1)) # escalamos los valores entre 0 y 1
scaler_X.fit(X_train) # entrenamos los valores quitandole la variable clase

X_train = scaler_X.transform(X_train)
X_test = scaler_X.transform(X_test)

In [ ]:
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS
from sklearn.neighbors import KNeighborsClassifier
import time
min_features = 3
max_features = 5

In [ ]:
knn = KNeighborsClassifier(n_neighbors=11)

efs = EFS(
    knn, 
    min_features=min_features, 
    max_features=max_features, # Evalúa hasta el total de columnas
    scoring='accuracy',
    cv=5,
    print_progress=True
)

efs = efs.fit(X_train, y_train)

print('Mejor combinación de índices:', efs.best_idx_)
print('Mejor puntaje:', efs.best_score_)

X_train_efs = efs.transform(X_train)
X_test_efs = efs.transform(X_test)